# UniMCP — Python quickstart

`unimcp` is a Cython extension over the UniMCP C ABI, shipped as a
self-contained wheel: the native library travels inside the package, so
installing it needs neither Nim nor a compiler.

```
pip install lituus-unimcp
```

UniMCP is a Model Context Protocol server, as a library. It owns the protocol —
JSON-RPC framing, the handshake, negotiation, the tool registry, dispatch — and
none of what makes a server yours.

CI executes this notebook against the wheel the release actually publishes, so
an output below that stops matching fails the build.

## The API

In [1]:
import json

import unimcp

unimcp.version(), unimcp.abi_version()

('0.1.0', 1)

## A server

Three things: what the server *is*, what tools it offers, and the callable that
runs them. The first two are the JSON documents the protocol itself defines.

In [2]:
def handler(name, arguments):
    if name != "shout":
        raise KeyError(name)
    return {"shouted": arguments["text"].upper()}


server = unimcp.Server(
    info={"name": "quickstart", "title": "Quickstart", "version": "1.0.0",
          "description": "One tool, shouted back",
          "latestProtocol": "2025-11-25",
          "supportedProtocols": ["2025-11-25", "2024-11-05"]},
    tools=[{"name": "shout", "title": "Shout",
            "description": "Upper-case a string",
            "inputSchema": {"type": "object",
                            "properties": {"text": {"type": "string"}},
                            "required": ["text"]},
            "readOnlyHint": True, "idempotentHint": True}],
    handler=handler)
type(server).__name__

'Server'

## The handshake

A client sends `initialize`, the server answers with the version both sides
will use, and the client acknowledges. Until it does, nothing else is served.

In [3]:
def send(method, params=None, id=None):
    message = {"jsonrpc": "2.0", "method": method}
    if id is not None:
        message["id"] = id
    if params is not None:
        message["params"] = params
    reply = server.handle(json.dumps(message))
    return json.loads(reply) if reply else None


send("initialize", {"protocolVersion": "2024-11-05", "capabilities": {},
                    "clientInfo": {"name": "quickstart", "version": "1"}}, id=1)

{'jsonrpc': '2.0',
 'id': 1,
 'result': {'protocolVersion': '2024-11-05',
  'capabilities': {'tools': {'listChanged': False}},
  'serverInfo': {'name': 'quickstart',
   'title': 'Quickstart',
   'version': '1.0.0',
   'description': 'One tool, shouted back'},
  'instructions': ''}}

The server answered with `2024-11-05`: the client asked for a
version it supports, so that is the one in force. A version it does not know
is answered with its own latest instead of a refusal.

A notification carries no `id` and gets no reply — `handle` returns `None`.

In [4]:
send("notifications/initialized")

## Discovery and dispatch

In [5]:
send("tools/list", id=2)

{'jsonrpc': '2.0',
 'id': 2,
 'result': {'tools': [{'name': 'shout',
    'title': 'Shout',
    'description': 'Upper-case a string',
    'inputSchema': {'type': 'object',
     'properties': {'text': {'type': 'string'}},
     'required': ['text']},
    'annotations': {'readOnlyHint': True,
     'destructiveHint': False,
     'idempotentHint': True,
     'openWorldHint': False}}]}}

In [6]:
send("tools/call",
     {"name": "shout", "arguments": {"text": "hello"}}, id=3)

{'jsonrpc': '2.0',
 'id': 3,
 'result': {'content': [{'type': 'text', 'text': '{"shouted":"HELLO"}'}],
  'structuredContent': {'shouted': 'HELLO'},
  'isError': False}}

## When a call fails

A tool that fails is not a broken connection. MCP reports it *in the result*,
with `isError` set, so a model can read the failure and try something else.

In [7]:
send("tools/call", {"name": "absent", "arguments": {}}, id=4)

{'jsonrpc': '2.0',
 'id': 4,
 'result': {'content': [{'type': 'text',
    'text': '{"error":"tool call failed: absent"}'}],
  'structuredContent': {'error': 'tool call failed: absent'},
  'isError': True}}

The exception the handler raised is kept rather than dropped:

In [8]:
repr(server.last_error)

"KeyError('absent')"

## Errors that are the protocol's, not a tool's

A malformed line, a method that does not exist, a call before the handshake:
each has its own JSON-RPC code.

In [9]:
[json.loads(server.handle("{")),
 send("nope", id=5)]

[{'jsonrpc': '2.0',
  'id': None,
  'error': {'code': -32700, 'message': 'Parse error'}},
 {'jsonrpc': '2.0',
  'id': 5,
  'error': {'code': -32601, 'message': 'Method not found'}}]

## A description that cannot work

Refused at construction, where the mistake is, rather than at the first
call.

In [10]:
try:
    unimcp.Server(info={}, tools=[], handler=handler)
except ValueError as exc:
    print("ValueError:", exc)

ValueError: supportedProtocols must be a JSON array


## The C ABI underneath

The same engine is reachable from anything that speaks C — two JSON documents
and a function pointer:

```c
void *unimcp_server_new(const char *info, const char *tools,
                        unimcp_tool_handler handler, void *user_data);
const char *unimcp_server_handle(void *server, const char *line);
```

There a failure is a `NULL` return with the reason in `unimcp_last_error`,
because an exception must never unwind across an ABI boundary.

See `include/UniMCP.h`, and the book for the full picture.